# Experiment To Validate Normalization Tokenizer Choice

Before computing WER, every reference and hypothesis is passed through Whisper's English text normaliser
(lowercasing, removing punctuation symbols, converting number words to digits, expanding contractions and
standardising British/American spelling). The normaliser is accessed through a Hugging Face `WhisperTokenizer`,
which has to be loaded from a specific Whisper checkpoint.

The normalisation code is shared between checkpoints, but each checkpoint ships its own spelling-mapping file,
so in principle the choice of checkpoint could change the normalised text and therefore the WER. This notebook
checks whether it does, by normalising the same references and transcripts with the tokenizer from eight
checkpoints:

* `tiny`, `base`, `small` and `medium`
* the multilingual and English-only (`.en`) version of each

If every tokenizer gives identical output, a single fixed tokenizer can be used for all scoring without affecting
the results.

## Imports

In [6]:
from pathlib import Path

import pandas as pd

from src.asr.normalize import (
    get_tokenizer,
    normalize_text,
)

from src.asr.metrics import compute_wer

import os

# Avoids hugging face warnings for Windows not using symlink
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

## Load Experiment Manifest and Transcriprion Results

In [7]:
RUN_DIR = Path(
    "../results/asr/runs/analysis_base_long"
)

manifest = pd.read_csv(
    RUN_DIR / "manifest_snapshot.csv"
)

transcriptions = pd.read_csv(
    RUN_DIR / "transcriptions.csv"
)

# Only uses successful transcriptions
transcriptions = transcriptions.loc[
    transcriptions["status"] == "ok"
].copy()

### Combine Results
Each row contains:
* sample_id
* dataset
* reference_transcript
* transcription_raw

In [8]:
data = manifest[
    [
        "sample_id",
        "dataset",
        "reference_transcript",
    ]
].merge(
    transcriptions[
        [
            "sample_id",
            "transcription_raw",
        ]
    ],
    on="sample_id",
    how="inner",
    validate="one_to_one",
)

## Define Tokenizer Models for the Experiment

In [9]:
NORMALIZER_MODELS = {
    "tiny": "openai/whisper-tiny",
    "tiny_en": "openai/whisper-tiny.en",

    "base": "openai/whisper-base",
    "base_en": "openai/whisper-base.en",

    "small": "openai/whisper-small",
    "small_en": "openai/whisper-small.en",

    "medium": "openai/whisper-medium",
    "medium_en": "openai/whisper-medium.en",
}

## Normalize Each Reference and Transcript Using Each Tokenizer

Each reference and hypothesis is normalised with every tokenizer using the same `normalize_text` function as the
scoring pipeline, so the only thing that changes between columns is the tokenizer.

In [10]:
for name, model_name in NORMALIZER_MODELS.items():

    print(f"Testing {model_name}...")

    tokenizer = get_tokenizer(
        model_name,
        revision=None
    )

    data[f"reference_norm_{name}"] = (
        data.apply(
            lambda row:
                normalize_text(
                    row["reference_transcript"],
                    row["dataset"],
                    tokenizer=tokenizer,
                )[0],
            axis=1,
        )
    )

    data[f"hypothesis_norm_{name}"] = (
        data.apply(
            lambda row:
                normalize_text(
                    row["transcription_raw"],
                    row["dataset"],
                    tokenizer=tokenizer,
                )[0],
            axis=1,
        )
    )

Testing openai/whisper-tiny...
Testing openai/whisper-tiny.en...
Testing openai/whisper-base...
Testing openai/whisper-base.en...
Testing openai/whisper-small...
Testing openai/whisper-small.en...
Testing openai/whisper-medium...
Testing openai/whisper-medium.en...


# Comparison 
Using whisper-small as the baseline

In [11]:
BASELINE = "small"

In [12]:
comparison_results = []

for name in NORMALIZER_MODELS:

    if name == BASELINE:
        continue

    reference_different = (
        data[f"reference_norm_{name}"]
        != data[
            f"reference_norm_{BASELINE}"
        ]
    )

    hypothesis_different = (
        data[f"hypothesis_norm_{name}"]
        != data[
            f"hypothesis_norm_{BASELINE}"
        ]
    )

    comparison_results.append(
        {
            "normalizer": name,
            "n_samples": len(data),

            "reference_differences":
                reference_different.sum(),

            "hypothesis_differences":
                hypothesis_different.sum(),
        }
    )

In [13]:
comparison_summary = pd.DataFrame(
    comparison_results
)

comparison_summary

,normalizer,n_samples,reference_differences,hypothesis_differences
0,tiny,580,0,0
1,tiny_en,580,0,0
2,base,580,0,0
3,base_en,580,0,0
4,small_en,580,0,0
5,medium,580,0,0
6,medium_en,580,0,0


**Analysis:** 

For all 580 recordings, every tokenizer produced exactly the same normalised reference and exactly
the same normalised hypothesis as the `whisper-small` baseline (0 differences for every checkpoint). This holds for
every model size and for both the multilingual and English-only versions.

### Differences

In [14]:
different = data.loc[
    (
        data["reference_norm_tiny"]
        != data["reference_norm_small"]
    )
    |
    (
        data["hypothesis_norm_tiny"]
        != data["hypothesis_norm_small"]
    )
]

different[
    [
        "sample_id",
        "dataset",
        "reference_transcript",
        "reference_norm_tiny",
        "reference_norm_small",
        "transcription_raw",
        "hypothesis_norm_tiny",
        "hypothesis_norm_small",
    ]
]

if different.empty:
    print("No differences between models")
else:
    print(f"Found {len(different)} differences:")
    display(different)

No differences between models


**Analysis:** 

Because the normalised text is identical, the WER is also identical for every recording under every
tokenizer (0 WER differences). The tokenizer choice therefore has no effect on any result in the study.

### WER Comparison

In [15]:
for name in NORMALIZER_MODELS:

    data[f"wer_{name}"] = data.apply(
        lambda row:
            compute_wer(
                row[f"reference_norm_{name}"],
                row[f"hypothesis_norm_{name}"],
            ).wer,
        axis=1,
    )

In [16]:
for name in NORMALIZER_MODELS:

    n_different = (
        data[f"wer_{name}"]
        != data["wer_small"]
    ).sum()

    print(
        f"{name}: "
        f"{n_different} WER differences"
    )

tiny: 0 WER differences
tiny_en: 0 WER differences
base: 0 WER differences
base_en: 0 WER differences
small: 0 WER differences
small_en: 0 WER differences
medium: 0 WER differences
medium_en: 0 WER differences


## Test Assertion 
Checks every normalizer gets the exact same result for every tested sample.

In [17]:
for name in NORMALIZER_MODELS:

    assert (
        data[f"reference_norm_{name}"]
        == data["reference_norm_small"]
    ).all()

    assert (
        data[f"hypothesis_norm_{name}"]
        == data["hypothesis_norm_small"]
    ).all()

## Conclusion
* Normalization was evaluated using the Whisper tiny, base, small, and medium tokenizers across both the reference transcripts and ASR hypotheses in the full analysis dataset. 
* No normalization differences were observed across the tested Whisper checkpoint sizes or between multilingual and English-only variants for the evaluated study data. 
* Therefore, the choice of tested Whisper tokenizer did not affect normalization or resulting WER for the evaluated analysis data.
* A single fixed Whisper tokenizer, `openai/whisper-small`, was therefore used for all subsequent scoring. 
* The tokenizer revision and the Transformers package version are pinned and recorded with each scoring run to ensure reproducibility.

## Record the Tokenizer Revision

The exact revision (commit hash) of the `openai/whisper-small` tokenizer is retrieved so it can be pinned in
`normalize.py`. This keeps the normalisation fixed even if the checkpoint is updated on Hugging Face later.

In [18]:
from huggingface_hub import HfApi

api = HfApi()

info = api.model_info(
    "openai/whisper-small"
)

print(info.sha)

973afd24965f72e36ca33b3055d56a652f456b4d
